In [ ]:
# Make sure the notebook can import the modules:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

In [ ]:
import re
import pandas as pd
import pdfplumber


In [ ]:
from plausibility import run_checks
from aggregate import aggregate
from query import select, distinct

## Add a publication description table (including sample description) to the schema:

### Load data: 

In [ ]:
ROOT = Path.cwd().parent
df = pd.read_csv(ROOT / "data" / "all_consolidated_corr.csv").drop(columns=['Unnamed: 0'])

In [ ]:
df.head()

### Create publication table and define data types:

In [ ]:
pub_info_table = pd.DataFrame({
    'pub_id': pd.Series(dtype='int'),
    'publication': pd.Series(dtype='str'),
    'sample_information': pd.Series(dtype='str'),
    'sample_with_mental_illness': pd.Series(dtype='int'),
    'sample_with_schizophrenia': pd.Series(dtype='int')
})

### Add data to publication table: 

In [ ]:
# Set the publication index:
pub_info_table['pub_id'] = list(range(0, len(set(df.publication))))
# Fill publication column with publication names (ordered alphabetically, however
# alphabetical order will not necessarily be maintained when adding further data 
# to the data base): 
pub_list = list(set(df.publication))
pub_list.sort()
pub_info_table['publication'] = pub_list
pub_info_table

In [ ]:
# Map source file names to publication names:
pub_source_file_mapping = df[['publication', 'source_file']].drop_duplicates().reset_index(drop=True).copy()
pub_source_file_mapping 

In [ ]:
# Make a dictionary mapping source file names to sample descriptions 
# (source file names are easier for keeping track when manually 
# creating the dictionary
# because they contain the name of the respective scale): 
sample_descriptions = {
   'PSYRATS_Favrod_et_al_2012_long.csv': 'schizophrenia or schizoaffective disorders',
    'PSYRATS_Haddock_et_al_1999_long.csv': ('All patients met DSM-III-R criteria '
                                            'for schizophrenia; schizophrenia; paranoid type '
                                            'and schizoaﬀective disorder (American Psychiatric Association, 1987).'),
    'PSYRATS_Woodward_et_al_2014_long.csv': ('schizophrenia: 31.5%; schizoaffective: 3.8%; psychosis '
                                            'NOS: 1.9%; first-episode psychosis: 5.0%; at-risk mental '
                                            'state: 3.8%; and seeking treatment for psychosis: 53.8%.'),
    'CTQ_SF_Hagborg_et_al_2022_long.csv': ('Data from the LoRDIA programme was used. LoRDIA '
                                            '(Longitudinal Research on Development In Adolescence) '
                                            'followed adolescents of two year-cohorts in four small and '
                                            'medium-sized municipalities (10,000-38,000 inhabitants) in '
                                            'the south of Sweden.'),
    'CTQ_SF_GarciaFernandez_et_al_2024_long.csv': ('208 adolescents with suicide attempts: ' 
                                            'All patients were recruited at the psychiatric ' 
                                            'emergence department from seven University Hospitals '
                                            'across Spain.'),
    'DERS_Giromini_et_al_2012_long.csv': ('Sample comprised of Italian psychology '
                                            'students at the Univer-sity of Milano-Bicocca.'),
    'DERS_GratzRoemer_2004_long.csv': ('Questionnaire packets were distributed to 479 students from undergraduate '
                                            'psychology courses offered at the University of Massachusetts Boston. '
                                            'Of these, 373 packets were returned 16 participants '
                                            'were excluded from the following analyses. '
                                            'The final sample of 357 participants ranged in age '
                                            'from 18 to 55 years. Seventy-three percent (n =260) of these partici'
                                            'pants were female.'),
    'DERS_Neumann_et_al_2010_long.csv': ('students from undergraduate ' 
                                            'psychology courses offered at the University of Massachusetts Boston '
                                            'students at a school for secondary education, includ'
                                            'ing Atheneum (60.3%), Gymnasium (21.5%), and HAVO (a '
                                            'Dutch acronym for “higher general secondary education”; '
                                            '18.2%), in Amsterdam, the Netherlands'),
    'DES-T_Giesbrecht_et_al_2007_long.csv': ('a convenience sample including '
                                            '930 undergraduate students enrolled at Maastricht University, '
                                            '20 healthy adult women, 22 patients with schizophrenia, 20 '
                                            'patients with borderline personality disorder, 19 patients with '
                                            'mood disorder without psychosis, and 55 women with a '
                                            'history of CSA.'),
    'DES-T_Levin_et_al_2003_long.csv': ('Participants were volunteers in the New York City met'
                                            'ropolitan area who were recruited by 1st-year doctoral '
                                            'graduate students in clinical psychology completing a '
                                            'psychological assessment practicum.'),
    'DES-T_Modestin_et_al_2004_long.csv': ('The non-clinical sample was composed of 276 '
                                            'non-patients, all medical students in their third and '
                                            'fourth years of study; The clinical sample comprised ' 
                                            '207 consecutively admitted psychiatric inpatients ' 
                                            'from the following principal ICD-10 '
                                            'diagnostic categories: substance use disorder,33%; '
                                            'schizophrenia spectrum disorder, 36%; affective '
                                            'disorder, 15%; neurotic, stress or somatoform dis'
                                            'order, 11%; and personality disorder, 5%.'),
    'DES-T_Spitzer_et_al_2014_long.csv': ('Die Daten wurden im Frühjahr 2014 im Rahmen einer bevölke'
                                            'rungsrepräsentativen Befragung erhoben, ' 
                                            'Die Zufallsauswahl der Haushalte erfolgte computerisiert nach '
                                            'der Random-Route-mit-Startadressen-Methode,')
}

In [ ]:
# Add sample descriptions and mental illness and schizophrenia flags to the dataframe:
for key, description in sample_descriptions.items():
    print('\n')
    print('key:')
    print(key)
    map_df = pub_source_file_mapping

    map_select = map_df[map_df.source_file==key]
    print('description:')
    print(description)
    df_select = pub_info_table[pub_info_table.publication == map_select.publication.iloc[0]]
    print('df_select:')
    print(df_select)
    
    index_number = df_select.index[0]
    
    # Set value in sample_information:
    pub_info_table.loc[index_number, 'sample_information'] = description
    # Set values for sample_with_schizophrenia and sample_with_mental_illness. 
    # CAVE! The conditions are not universally valid, and verification by
    # visual inspection is required!
    if ('schizo' in description) :
        print('schizophrenia YES')
        print(description)
        pub_info_table.loc[index_number, 'sample_with_schizophrenia'] = 1
        pub_info_table.loc[index_number, 'sample_with_mental_illness'] = 1
    elif (('psychi' in description)):
        print('mental illness YES')
        print(description)
        pub_info_table.loc[index_number, 'sample_with_schizophrenia'] = 0
        pub_info_table.loc[index_number, 'sample_with_mental_illness'] = 1
    else:
        print('NO')
        pub_info_table.loc[index_number, 'sample_with_schizophrenia'] = 0
        pub_info_table.loc[index_number, 'sample_with_mental_illness'] = 0

    
    

In [ ]:
# Have a look:
pub_info_table

In [ ]:
pub_info_table.columns

In [ ]:
pub_info_table.head(3)

In [ ]:
pub_info_table.sample_information[1]

### Add publication id to the fact table dataframe:

In [ ]:
df.head()

In [ ]:
# Loop through fact table dataframe and get the publication name.
# For each row get the correct id, sample_with_mental_illness
# and correct sample_with_schizophrenia values
# from the pub_info_table by publication name
# and add the values to the prepared empty lists: 
pub_ids = []
with_mental_illness_samples = []
with_schizophrenia_samples = []
for index, row in df.iterrows():
    # print(index)
    # print(type(row))
    # print(type(row.publication))
    # print(row.publication)
    # print(row.publication == 'Garcia-Fernandez et al. 2024')
    # print(sum(pub_info_table.publication == row.publication))
    selection_bools = pub_info_table.publication == row.publication
    pub_info_slice = pub_info_table[selection_bools]
    # print(type(pub_info_slice))
    # print(type(pub_info_slice.pub_id))
    # print(pub_info_slice.pub_id.shape)
    # print(type(pub_info_slice.pub_id.iloc[0]))
    # print(pub_info_slice.pub_id.iloc[0])
    pub_id = pub_info_slice.pub_id.iloc[0]
    pub_ids.append(pub_id)
    # print(row.sample_type)
    # print(type(row.sample_type))
    if row.sample_type == 'patients':
        # print('yes')
        # print(pub_info_slice.sample_with_mental_illness.iloc[0])
        with_mental_illness = pub_info_slice.sample_with_mental_illness.iloc[0]
        with_mental_illness_samples.append(with_mental_illness)

        with_mental_illness = pub_info_slice.sample_with_schizophrenia.iloc[0]
        with_schizophrenia_samples.append(with_mental_illness)
    else:
        # print('no')
        with_mental_illness_samples.append(0)
        with_schizophrenia_samples.append(0)
        

In [ ]:
print(len(pub_ids))
print(df.shape)

In [ ]:
# Add pub_ids, with_mental_illness_samples, and with_schizophrenia_samples lists to dataframe: 
df['pub_id'] = pub_ids
df['sample_with_mental_illness'] = with_mental_illness_samples
df['sample_with_schizophrenia'] = with_schizophrenia_samples

In [ ]:
df.head()

In [ ]:
df_patients_only = df[df.sample_type=='patients']

set(df_patients_only[df_patients_only.sample_with_mental_illness==0].source_file)

In [ ]:
pub_info_table[pub_info_table.sample_with_mental_illness==0]

In [ ]:
set(df[df.sample_with_mental_illness==0].source_file)

### Make spot checks if id values were attributed correctly:

In [ ]:
test_pub_id = 2
pub_info_table.iloc[test_pub_id,]


In [ ]:
pub_info_table.iloc[test_pub_id,].sample_information

In [ ]:
test_pub = pub_info_table.iloc[test_pub_id,].publication
print(set(df[df.publication==test_pub].pub_id))
print(test_pub)
check_id = list(set(df[df.publication==test_pub].pub_id))[0]
test_pub_id == check_id

In [ ]:
list(df[df.publication==test_pub].sample_with_mental_illness.drop_duplicates())

In [ ]:
list(df[df.publication==test_pub].sample_with_schizophrenia.drop_duplicates())

In [ ]:
list(df[df.publication==test_pub].sample_type.drop_duplicates())

### Automate checks: 

In [ ]:
test_pub_id = 3
test_pub_ids = list(range(0,12))
checks = []
for test_pub_id in test_pub_ids: 
    test_pub = pub_info_table.iloc[test_pub_id,].publication
    check_id = list(set(df[df.publication==test_pub].pub_id))[0]
    #test_pub_id == check_id
    checks.append(test_pub_id == check_id)

In [ ]:
# Expected value: True
sum(checks) == pub_info_table.shape[0]

In [ ]:
test_pub_id = 3
test_pub_ids = list(range(0,12))
checks = []
values_length_ill_comp = []
values_length_not_ill_comp = []
patient_mental_ill_values_check = []
not_ill_value_check = []
for test_pub_id in test_pub_ids: 
    test_sample_illness = pub_info_table.iloc[test_pub_id,].sample_with_mental_illness
    df_slice = df[df.pub_id == test_pub_id].copy()
    print(df_slice.shape)
    print(type(df_slice))
    patient_slice = df_slice[df_slice.sample_type=='patients']
    if test_sample_illness == 1: 
        print(set(list(df_slice.sample_with_mental_illness)))
        print(df_slice.sample_with_mental_illness.drop_duplicates().shape[0])
        mental_values_len = df_slice.sample_with_mental_illness.drop_duplicates().shape[0]
        sample_values_len = df_slice.sample_type.drop_duplicates().shape[0]
        values_length_ill_comp.append(mental_values_len == sample_values_len)
        print(df_slice[df_slice.sample_type=='patients'].sample_with_mental_illness.drop_duplicates())
        mental_values_patients = df_slice[df_slice.sample_type=='patients'].sample_with_mental_illness.drop_duplicates()
        if mental_values_patients.shape[0] == 1:
            print(int(mental_values_patients.iloc[0])==1)
            patient_mental_ill_values_check.append(int(mental_values_patients.iloc[0])==1)
        else:
            patient_mental_ill_values_check.append(False)
    else:
        mental_values_len = df_slice.sample_with_mental_illness.drop_duplicates().shape[0]
        values_length_not_ill_comp.append(mental_values_len == 1)
        print(mental_values_len)
        print(type(mental_values_len))
        if (mental_values_len == 1): 
            not_ill_value_check.append(int(df_slice.sample_with_mental_illness.drop_duplicates().iloc[0]) == 0)
        else:
            not_ill_value_check.append(False)
    #break


In [ ]:
df_slice.sample_with_mental_illness.drop_duplicates().shape[0]

In [ ]:
print(set(values_length_ill_comp))
print(set(values_length_not_ill_comp))
print(set(patient_mental_ill_values_check))
print(set(not_ill_value_check))

In [ ]:
mental_values_patients.shape[0]

### Have a look:

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
import os
os.getcwd()

In [ ]:
ROOT

In [ ]:
os.listdir(ROOT/'data'/'consolidated')

In [ ]:
os.listdir(ROOT)

In [ ]:
os.listdir(ROOT/'data')

### Save the new dataframe:

In [ ]:
df.to_csv(ROOT/'data'/'all_consolidated_corr.csv', index=False)

### Save the pub_info_table:

In [ ]:
pub_info_table.to_csv(ROOT/'data'/'pub_info_table.csv', index=False)